In [8]:
import scipy.stats as sc

In [3]:
import pandas as pd

In [20]:
import numpy as np

In [28]:
import statsmodels.formula.api as smf

In [9]:
mcd_df = pd.read_csv('/Users/W.Zhouhan/GitHub/EC2C3/pset3/mcdonalds.csv')
mcd_df.head(5)

,price,location,num_city,city,address,zipcode,latitude,longitude,mcwebsite,county,state_name
0,7.70,Urban,1,Eagle River,11808 Old Glenn Hwy,99577,61.327018,-149.57134,https://www.mcdonalds.com/us/en-us/location/ak...,Anchorage Borough,Alaska
1,7.80,Urban,13,Anchorage,2601 Tudor Rd,99508,61.181320,-149.83240,https://www.mcdonalds.com/us/en-us/location/ak...,Anchorage Borough,Alaska
2,7.70,Urban,13,Anchorage,255 Muldoon Rd,99504,61.221444,-149.73251,https://www.mcdonalds.com/us/en-us/location/ak...,Anchorage Borough,Alaska
3,7.75,Urban,13,Anchorage,800 W Northern Lights Blvd,99503,61.194617,-149.89863,https://www.mcdonalds.com/us/en-us/location/ak...,Anchorage Borough,Alaska
4,7.85,Urban,1,Palmer,600 Daron Dr,99645,61.598982,-149.12364,https://www.mcdonalds.com/us/en-us/location/ak...,Matanuska-Susitna Borough,Alaska


In [13]:
# Create separate series for urban and rural
mcd_urban = mcd_df.loc[mcd_df['location'] == 'Urban', 'price']
mcd_rural = mcd_df.loc[mcd_df['location'] == 'Rural', 'price']

# We assume Welch's t-test, as each group's variance is most likely different
results = sc.ttest_ind(mcd_urban, mcd_rural, equal_var=False)
print(f"t({results.df:.1f}) = {results.statistic:.3f}, p = {results.pvalue:.4f}")

t(300.4) = 1.400, p = 0.1625


(a) Since we have p-value = 0.162 > 0.05, we fail to reject the null hypothesis at 95% confidence interval.

In [19]:
# Create separate dfs for urban and rural
mcd_urban_df = mcd_df.loc[mcd_df['location'] == 'Urban']

mcd_rural_df = mcd_df.loc[mcd_df['location'] == 'Rural']

# We assume Welch's t-test, as each group's variance is most likely different
results = sc.ttest_ind(mcd_urban_df['num_city'], mcd_rural_df['num_city'], equal_var=False)
print(f"t({results.df:.1f}) = {results.statistic:.3f}, p = {results.pvalue:.10f}")

t(3990.4) = 30.863, p = 0.0000000000


(a) We do this without dedupping, assuming that's what the question wants. As p-value is clearly smaller than 0.05, we reject the null.

In [22]:
# Find median num_city
median_num_city = np.median(mcd_df['num_city'])
print(median_num_city)

num_city_above = mcd_df.loc[mcd_df['num_city'] > median_num_city, 'price']
num_city_below = mcd_df.loc[mcd_df['num_city'] <= median_num_city, 'price']

# We assume Welch's t-test, as each group's variance is most likely different
results = sc.ttest_ind(num_city_above, num_city_below, equal_var=False)
print(f"t({results.df:.1f}) = {results.statistic:.3f}, p = {results.pvalue:.10f}")




4.0
t(3793.7) = -0.853, p = 0.3938289774


(b) We do this without dedupping, assuming that's what the question wants. As p-value is greater than 0.05, we fail to reject the null.

In [24]:
# We create a new variable that is num_city_below - 0.2
num_city_below_transformed = num_city_below - 0.2

# We assume Welch's t-test, as each group's variance is most likely different
results = sc.ttest_ind(num_city_above, num_city_below_transformed, equal_var=False)
print(f"t({results.df:.1f}) = {results.statistic:.3f}, p = {results.pvalue:.10f}")

t(3793.7) = 6.985, p = 0.0000000000


(c) We do this without dedupping, assuming that's what the question wants. As p-value is smaller than 0.05, we reject the null.

In [26]:
beta_one = mcd_df['price'].cov(mcd_df['num_city']) / mcd_df['num_city'].var()
beta_naught = mcd_df['price'].mean() - beta_one * mcd_df['num_city'].mean()
print(f'beta_one={beta_one}, beta_naught={beta_naught}')

beta_one=0.001640747405265988, beta_naught=6.705489211708952


In [32]:
# Get residuals by using the beta values calculated earlier
mcd_df['residual'] = mcd_df['price'] - (mcd_df['num_city'] * beta_one) - beta_naught

model = smf.ols('price ~ num_city', data=mcd_df).fit()
print(model.summary())



                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     5.765
Date:                Wed, 21 Oct 2026   Prob (F-statistic):             0.0164
Time:                        18:44:46   Log-Likelihood:                -4794.8
No. Observations:                4000   AIC:                             9594.
Df Residuals:                    3998   BIC:                             9606.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.7055      0.015    452.675      0.0

In [31]:
# Compare residuals
print(np.allclose(mcd_df['residual'], model.resid))

# See the largest gap between the two
print((mcd_df['residual'] - model.resid).abs().max())

True
1.7763568394002505e-15


In [33]:
# Run residual on num_city as regression
model_error = smf.ols('residual ~ num_city', data=mcd_df).fit()
print(model_error.summary())

                            OLS Regression Results                            
Dep. Variable:               residual   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                 7.061e-13
Date:                Wed, 21 Oct 2026   Prob (F-statistic):               1.00
Time:                        18:48:15   Log-Likelihood:                -4794.8
No. Observations:                4000   AIC:                             9594.
Df Residuals:                    3998   BIC:                             9606.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  -6.557e-16      0.015  -4.43e-14      1.0

In [35]:
# Standardisation
mcd_df['price_standardised'] = mcd_df['price'] / mcd_df['price'].std()
mcd_df['num_city_standardised'] = mcd_df['num_city'] / mcd_df['num_city'].std()

model_error = smf.ols('price_standardised ~ num_city_standardised', data=mcd_df).fit()
print(model_error.summary())

                            OLS Regression Results                            
Dep. Variable:     price_standardised   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     5.765
Date:                Wed, 21 Oct 2026   Prob (F-statistic):             0.0164
Time:                        18:52:42   Log-Likelihood:                -5672.4
No. Observations:                4000   AIC:                         1.135e+04
Df Residuals:                    3998   BIC:                         1.136e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 8.35